# 04 — Velocity-State Kalman Control

**Repository:** `quantum-hardware-company`  
**Directory:** `control-stack/`

This notebook upgrades predictive control by adding a velocity state to the Kalman estimator.

## Purpose

Notebook 03 showed that naive predictive Kalman underperformed because the state model was too simple.

This notebook asks:

> Can prediction improve if the estimator models both drift and drift rate?

## State model

Instead of estimating only drift:

```text
x = [drift]
```

we estimate:

```text
x = [drift, drift_rate]
```

This lets the controller predict the next drift value before the next calibration block is measured.

## Policies compared

- no compensation
- moving average
- scalar Kalman
- velocity-state Kalman
- velocity-state predictive Kalman
- oracle baseline

## Principle

Prediction requires dynamics.

A predictive controller must model how drift changes, not only where drift is now.


## Velocity-state model

For each parameter, use a constant-velocity state-space model:

$$
x_k =
\begin{bmatrix}
d_k \\
\dot{d}_k
\end{bmatrix}
$$

with transition:

$$
x_k =
\begin{bmatrix}
1 & \Delta t \\
0 & 1
\end{bmatrix}
x_{k-1} + w_k
$$

and measurement:

$$
z_k =
\begin{bmatrix}
1 & 0
\end{bmatrix}
x_k + v_k
$$

where:

- $d_k$ is drift,
- $\dot{d}_k$ is drift rate,
- $z_k$ is measured drift from calibration,
- $w_k$ is process noise,
- $v_k$ is measurement noise.

The one-step prediction is:

$$
\hat{d}_{k+1|k} = \hat{d}_{k|k} + \Delta t\,\hat{\dot{d}}_{k|k}
$$


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
def rabi_model(t, A, Omega, B):
    return A * np.sin(0.5 * Omega * t) ** 2 + B


def fit_model(model, x, y, p0, bounds=(-np.inf, np.inf)):
    params, cov = curve_fit(model, x, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def phase_lock_metric(signal, target):
    dot = np.dot(signal, target)
    norm = np.linalg.norm(signal) * np.linalg.norm(target)
    if norm == 0:
        return np.nan
    return dot / norm


def moving_average(x, window=4):
    y = np.zeros_like(x, dtype=float)
    for i in range(len(x)):
        lo = max(0, i - window + 1)
        y[i] = np.mean(x[lo:i + 1])
    return y


def kalman_filter_1d(z, q, r, x0=None, p0=1.0):
    z = np.asarray(z, dtype=float)
    x_hat = np.zeros_like(z)
    p_hist = np.zeros_like(z)
    k_hist = np.zeros_like(z)

    x = z[0] if x0 is None else float(x0)
    p = float(p0)

    for i, measurement in enumerate(z):
        x_pred = x
        p_pred = p + q

        k_gain = p_pred / (p_pred + r)
        x = x_pred + k_gain * (measurement - x_pred)
        p = (1 - k_gain) * p_pred

        x_hat[i] = x
        p_hist[i] = p
        k_hist[i] = k_gain

    return x_hat, p_hist, k_hist


def kalman_filter_velocity_1d(z, q_pos, q_vel, r, dt=1.0, x0=None, P0=None):
    """2D constant-velocity Kalman filter for drift.

    State:
        x = [drift, drift_rate]

    Transition:
        F = [[1, dt],
             [0, 1]]

    Measurement:
        H = [[1, 0]]
    """
    z = np.asarray(z, dtype=float)

    F = np.array([[1.0, dt], [0.0, 1.0]])
    H = np.array([[1.0, 0.0]])
    Q = np.array([[q_pos, 0.0], [0.0, q_vel]])
    R = np.array([[r]])

    if x0 is None:
        if len(z) > 1:
            initial_rate = z[1] - z[0]
        else:
            initial_rate = 0.0
        x = np.array([[z[0]], [initial_rate]])
    else:
        x = np.array(x0, dtype=float).reshape(2, 1)

    if P0 is None:
        P = np.eye(2)
    else:
        P = np.array(P0, dtype=float)

    x_filt = np.zeros((len(z), 2))
    x_pred = np.zeros((len(z), 2))
    P_trace = np.zeros((len(z), 2, 2))
    K_trace = np.zeros((len(z), 2))

    for i, measurement in enumerate(z):
        # Predict
        x_prior = F @ x
        P_prior = F @ P @ F.T + Q

        # Update
        y = np.array([[measurement]]) - H @ x_prior
        S = H @ P_prior @ H.T + R
        K = P_prior @ H.T @ np.linalg.inv(S)

        x = x_prior + K @ y
        P = (np.eye(2) - K @ H) @ P_prior

        x_pred[i] = x_prior.ravel()
        x_filt[i] = x.ravel()
        P_trace[i] = P
        K_trace[i] = K.ravel()

    return x_filt, x_pred, P_trace, K_trace


def one_step_velocity_prediction(x_filt, dt=1.0):
    """Prediction available for next block using filtered [drift, drift_rate]."""
    pred_next = x_filt[:, 0] + dt * x_filt[:, 1]
    pred = np.zeros_like(pred_next)
    pred[0] = x_filt[0, 0]
    pred[1:] = pred_next[:-1]
    return pred


def evaluate_policy(name, delta_omega_hat, delta_b_hat, true_delta_omega, true_delta_b):
    omega_controlled = target_Omega + true_delta_omega - delta_omega_hat
    b_controlled = np.clip(target_B + true_delta_b - delta_b_hat, 0, 0.3)

    target_signal_local = rabi_model(pulse_t, target_A, target_Omega, target_B)

    response_rmse = []
    phase_lock = []
    controlled_signals = []

    for k in range(len(true_delta_omega)):
        y = rabi_model(pulse_t, target_A, omega_controlled[k], b_controlled[k])
        controlled_signals.append(y)
        response_rmse.append(rmse(y, target_signal_local))
        phase_lock.append(phase_lock_metric(y, target_signal_local))

    response_rmse = np.array(response_rmse)
    phase_lock = np.array(phase_lock)
    controlled_signals = np.array(controlled_signals)

    return {
        "name": name,
        "omega_controlled": omega_controlled,
        "b_controlled": b_controlled,
        "omega_error": omega_controlled - target_Omega,
        "b_error": b_controlled - target_B,
        "omega_rmse": rmse(omega_controlled, np.full(len(omega_controlled), target_Omega)),
        "b_rmse": rmse(b_controlled, np.full(len(b_controlled), target_B)),
        "response_rmse": response_rmse,
        "response_rmse_mean": float(np.mean(response_rmse)),
        "response_rmse_max": float(np.max(response_rmse)),
        "phase_lock": phase_lock,
        "min_phase_lock": float(np.min(phase_lock)),
        "mean_phase_lock": float(np.mean(phase_lock)),
        "max_angle_degrees": float(np.max(np.degrees(np.arccos(np.clip(phase_lock, -1, 1))))),
        "controlled_signals": controlled_signals,
    }


## Generate synthetic drift environment

The environment matches notebooks 01–03 so policy comparisons remain consistent.


In [ ]:
n_blocks = 36
block_times = np.arange(n_blocks)
dt = 1.0

n_points_per_block = 140
pulse_t = np.linspace(0, 10, n_points_per_block)

target_A = 0.88
target_Omega = 2.45
target_B = 0.06

target_signal = rabi_model(pulse_t, target_A, target_Omega, target_B)

true_delta_Omega = (
    0.055 * np.sin(2 * np.pi * block_times / 21 + 0.4)
    + 0.018 * np.sin(2 * np.pi * block_times / 9)
)
true_delta_B = (
    0.0016 * block_times
    + 0.006 * np.sin(2 * np.pi * block_times / 14 + 0.2)
)

Omega_uncomp = target_Omega + true_delta_Omega
B_uncomp = target_B + true_delta_B
A_uncomp = target_A + 0.012 * np.sin(2 * np.pi * block_times / 17)

print(f"Target Omega = {target_Omega:.4f}")
print(f"Target B     = {target_B:.4f}")


## Simulate calibration measurements and fit each block


In [ ]:
Y_obs = []
Y_clean = []

for k in range(n_blocks):
    y_clean = rabi_model(pulse_t, A_uncomp[k], Omega_uncomp[k], B_uncomp[k])
    noise = 0.04 * np.random.randn(n_points_per_block)
    y_obs = np.clip(y_clean + noise, 0, 1)
    Y_clean.append(y_clean)
    Y_obs.append(y_obs)

Y_clean = np.array(Y_clean)
Y_obs = np.array(Y_obs)

fit_params = []
fit_stds = []

initial_guess = [0.85, 2.40, 0.05]
bounds = ([0.0, 0.0, 0.0], [1.2, 5.0, 0.3])

for k in range(n_blocks):
    params, cov = fit_model(
        rabi_model,
        pulse_t,
        Y_obs[k],
        p0=initial_guess,
        bounds=bounds,
    )
    fit_params.append(params)
    fit_stds.append(np.sqrt(np.diag(cov)))

fit_params = np.array(fit_params)
fit_stds = np.array(fit_stds)

Omega_est = fit_params[:, 1]
B_est = fit_params[:, 2]

delta_Omega_est = Omega_est - target_Omega
delta_B_est = B_est - target_B

print("Calibration fit complete.")


## Define baseline and velocity-state estimators


In [ ]:
# Baseline: no compensation
delta_Omega_none = np.zeros(n_blocks)
delta_B_none = np.zeros(n_blocks)

# Baseline: moving average
ma_window = 4
delta_Omega_ma = moving_average(delta_Omega_est, window=ma_window)
delta_B_ma = moving_average(delta_B_est, window=ma_window)

# Scalar Kalman from notebook 02
r_omega = float(np.median(fit_stds[:, 1] ** 2))
r_b = float(np.median(fit_stds[:, 2] ** 2))

q_omega_scalar = 8.0e-4
q_b_scalar = 1.0e-5

delta_Omega_scalar, p_Omega_scalar, k_Omega_scalar = kalman_filter_1d(
    delta_Omega_est, q=q_omega_scalar, r=r_omega, p0=1.0
)
delta_B_scalar, p_B_scalar, k_B_scalar = kalman_filter_1d(
    delta_B_est, q=q_b_scalar, r=r_b, p0=1.0
)

# Velocity-state Kalman
q_omega_pos = 2.0e-5
q_omega_vel = 2.0e-4
q_b_pos = 2.0e-6
q_b_vel = 1.0e-6

omega_state_filt, omega_state_pred, P_omega_vel, K_omega_vel = kalman_filter_velocity_1d(
    delta_Omega_est,
    q_pos=q_omega_pos,
    q_vel=q_omega_vel,
    r=r_omega,
    dt=dt,
)

b_state_filt, b_state_pred, P_b_vel, K_b_vel = kalman_filter_velocity_1d(
    delta_B_est,
    q_pos=q_b_pos,
    q_vel=q_b_vel,
    r=r_b,
    dt=dt,
)

delta_Omega_vel_filtered = omega_state_filt[:, 0]
delta_B_vel_filtered = b_state_filt[:, 0]

delta_Omega_vel_predictive = one_step_velocity_prediction(omega_state_filt, dt=dt)
delta_B_vel_predictive = one_step_velocity_prediction(b_state_filt, dt=dt)

# Oracle
delta_Omega_oracle = true_delta_Omega.copy()
delta_B_oracle = true_delta_B.copy()

print("Estimators ready.")


## Estimator comparison

The velocity-state filter estimates both drift and drift rate.

The predictive version uses the next-block forecast available from the filtered state.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(block_times, true_delta_Omega, linewidth=2, label="true drift")
plt.plot(block_times, delta_Omega_est, marker="o", linewidth=1, label="measured drift")
plt.plot(block_times, delta_Omega_ma, linewidth=2, linestyle="--", label="moving average")
plt.plot(block_times, delta_Omega_scalar, linewidth=2, linestyle=":", label="scalar Kalman")
plt.plot(block_times, delta_Omega_vel_filtered, linewidth=2, linestyle="-.", label="velocity-state filtered")
plt.plot(block_times, delta_Omega_vel_predictive, linewidth=2, label="velocity-state predictive")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Omega drift estimate")
plt.title("Velocity-state estimator comparison: Rabi frequency")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_01_omega_estimator_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_01_omega_estimator_comparison.pdf")

plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(block_times, true_delta_B, linewidth=2, label="true drift")
plt.plot(block_times, delta_B_est, marker="o", linewidth=1, label="measured drift")
plt.plot(block_times, delta_B_ma, linewidth=2, linestyle="--", label="moving average")
plt.plot(block_times, delta_B_scalar, linewidth=2, linestyle=":", label="scalar Kalman")
plt.plot(block_times, delta_B_vel_filtered, linewidth=2, linestyle="-.", label="velocity-state filtered")
plt.plot(block_times, delta_B_vel_predictive, linewidth=2, label="velocity-state predictive")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("offset drift estimate")
plt.title("Velocity-state estimator comparison: readout offset")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_02_offset_estimator_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_02_offset_estimator_comparison.pdf")

plt.show()


## Drift-rate estimates

Velocity-state Kalman exposes an interpretable drift-rate estimate.

This becomes the basis for predictive control.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(block_times, omega_state_filt[:, 1], marker="o", linewidth=1, label="Omega drift rate")
plt.plot(block_times, b_state_filt[:, 1], marker="o", linewidth=1, label="B drift rate")
plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("estimated drift rate")
plt.title("Velocity-state Kalman: drift-rate estimates")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_03_drift_rate_estimates.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_03_drift_rate_estimates.pdf")

plt.show()


## Evaluate control policies

Policies now include velocity-state filtered and velocity-state predictive control.


In [ ]:
policies = {
    "none": (delta_Omega_none, delta_B_none),
    "moving_average": (delta_Omega_ma, delta_B_ma),
    "scalar_kalman": (delta_Omega_scalar, delta_B_scalar),
    "velocity_filtered": (delta_Omega_vel_filtered, delta_B_vel_filtered),
    "velocity_predictive": (delta_Omega_vel_predictive, delta_B_vel_predictive),
    "oracle": (delta_Omega_oracle, delta_B_oracle),
}

policy_results = {}

for name, (delta_omega_hat, delta_b_hat) in policies.items():
    policy_results[name] = evaluate_policy(
        name,
        delta_omega_hat,
        delta_b_hat,
        true_delta_Omega,
        true_delta_B,
    )

for name, result in policy_results.items():
    print(
        f"{name:20s} | "
        f"response RMSE mean = {result['response_rmse_mean']:.6f} | "
        f"max = {result['response_rmse_max']:.6f} | "
        f"min phase-lock = {result['min_phase_lock']:.6f}"
    )


## Response-level error comparison


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    plt.plot(block_times, result["response_rmse"], marker="o", linewidth=1, label=name)

plt.xlabel("calibration block / lab time index")
plt.ylabel("RMSE vs target response")
plt.title("Velocity-state control: response-level error comparison")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_04_response_rmse_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_04_response_rmse_comparison.pdf")

plt.show()


## Parameter error comparison


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    if name == "oracle":
        continue
    plt.plot(block_times, result["omega_error"], marker="o", linewidth=1, label=name)

plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("Omega error from target")
plt.title("Velocity-state control: Omega error comparison")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_05_omega_error_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_05_omega_error_comparison.pdf")

plt.show()


In [ ]:
plt.figure(figsize=(11, 5))
for name, result in policy_results.items():
    if name == "oracle":
        continue
    plt.plot(block_times, result["b_error"], marker="o", linewidth=1, label=name)

plt.axhline(0, linewidth=1)
plt.xlabel("calibration block / lab time index")
plt.ylabel("offset error from target")
plt.title("Velocity-state control: offset error comparison")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_06_offset_error_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_06_offset_error_comparison.pdf")

plt.show()


## CGCS phase-lock stability


In [ ]:
threshold = 1 / np.sqrt(2)

plt.figure(figsize=(11, 5))
plt.axhline(threshold, linewidth=1, linestyle="--", label="45° threshold")

for name, result in policy_results.items():
    plt.plot(block_times, result["phase_lock"], marker="o", linewidth=1, label=name)

plt.ylim(0.68, 1.01)
plt.xlabel("calibration block / lab time index")
plt.ylabel("cosine similarity vs target")
plt.title("Velocity-state control: CGCS phase-lock stability")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_07_cgcs_stability_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_07_cgcs_stability_comparison.pdf")

plt.show()


## Worst-case block comparison

We choose the block where no compensation has the largest response error.


In [ ]:
worst_block = int(np.argmax(policy_results["none"]["response_rmse"]))

plt.figure(figsize=(11, 5))
plt.plot(pulse_t, target_signal, linewidth=3, label="target")

for name, result in policy_results.items():
    plt.plot(pulse_t, result["controlled_signals"][worst_block], linewidth=2, label=name)

plt.xlabel("time / pulse duration")
plt.ylabel("excited-state probability")
plt.title(f"Velocity-state control: worst-case block {worst_block}")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_08_worst_case_block_comparison.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_08_worst_case_block_comparison.pdf")

plt.show()


## Policy ranking


In [ ]:
ranking = sorted(policy_results.values(), key=lambda r: r["response_rmse_mean"])

policy_table = []
for rank, result in enumerate(ranking, start=1):
    policy_table.append({
        "rank": rank,
        "policy": result["name"],
        "omega_rmse": result["omega_rmse"],
        "b_rmse": result["b_rmse"],
        "response_rmse_mean": result["response_rmse_mean"],
        "response_rmse_max": result["response_rmse_max"],
        "min_phase_lock": result["min_phase_lock"],
        "mean_phase_lock": result["mean_phase_lock"],
        "max_angle_degrees": result["max_angle_degrees"],
        "all_blocks_phase_locked": bool(np.all(result["phase_lock"] >= threshold)),
    })

policy_table


In [ ]:
policy_names_ranked = [row["policy"] for row in policy_table]
response_means_ranked = [row["response_rmse_mean"] for row in policy_table]
response_max_ranked = [row["response_rmse_max"] for row in policy_table]

x = np.arange(len(policy_names_ranked))
width = 0.35

plt.figure(figsize=(11, 5))
bars_mean = plt.bar(x - width / 2, response_means_ranked, width, label="mean RMSE")
bars_max = plt.bar(x + width / 2, response_max_ranked, width, label="max RMSE")

plt.xticks(x, policy_names_ranked, rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("Velocity-state control: policy ranking")
plt.legend()

for bars in [bars_mean, bars_max]:
    for bar in bars:
        value = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.002,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )

plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_09_policy_ranking_summary.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_09_policy_ranking_summary.pdf")

plt.show()


## Velocity-state tuning sweep

Sweep velocity process noise for Ω to show the responsiveness/smoothing tradeoff.


In [ ]:
q_vel_values = np.logspace(-7, -2, 30)
sweep_response_rmse = []

for q_vel_test in q_vel_values:
    omega_state_test, _, _, _ = kalman_filter_velocity_1d(
        delta_Omega_est,
        q_pos=q_omega_pos,
        q_vel=q_vel_test,
        r=r_omega,
        dt=dt,
    )
    delta_omega_pred_test = one_step_velocity_prediction(omega_state_test, dt=dt)

    test_eval = evaluate_policy(
        "velocity_predictive_test",
        delta_omega_pred_test,
        delta_B_vel_predictive,
        true_delta_Omega,
        true_delta_B,
    )
    sweep_response_rmse.append(test_eval["response_rmse_mean"])

sweep_response_rmse = np.array(sweep_response_rmse)
best_q_vel_idx = int(np.argmin(sweep_response_rmse))
best_q_omega_vel = float(q_vel_values[best_q_vel_idx])

print(f"Current q_omega_vel = {q_omega_vel:.8f}")
print(f"Best swept q_omega_vel = {best_q_omega_vel:.8f}")
print(f"Best swept response RMSE = {sweep_response_rmse[best_q_vel_idx]:.6f}")


In [ ]:
plt.figure(figsize=(10, 5))
plt.semilogx(q_vel_values, sweep_response_rmse, marker="o", linewidth=1)
plt.axvline(q_omega_vel, linewidth=1, linestyle="--", label=f"current q_vel = {q_omega_vel:.1e}")
plt.axvline(best_q_omega_vel, linewidth=1, linestyle=":", label=f"best q_vel = {best_q_omega_vel:.1e}")
plt.xlabel("Omega velocity process variance")
plt.ylabel("mean response RMSE")
plt.title("Velocity-state tuning sweep")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/velocity_10_q_velocity_sweep.png", dpi=300)
plt.savefig(f"{FIG_DIR}/velocity_10_q_velocity_sweep.pdf")

plt.show()


## Save velocity-state control summary


In [ ]:
summary = {
    "n_blocks": int(n_blocks),
    "moving_average_window": int(ma_window),
    "scalar_kalman_q_omega": float(q_omega_scalar),
    "scalar_kalman_q_b": float(q_b_scalar),
    "velocity_q_omega_pos": float(q_omega_pos),
    "velocity_q_omega_vel": float(q_omega_vel),
    "velocity_q_b_pos": float(q_b_pos),
    "velocity_q_b_vel": float(q_b_vel),
    "kalman_r_omega": float(r_omega),
    "kalman_r_b": float(r_b),
    "phase_lock_threshold": float(threshold),
    "best_swept_q_omega_vel": float(best_q_omega_vel),
    "best_swept_q_omega_vel_response_rmse": float(sweep_response_rmse[best_q_vel_idx]),
    "ranking": policy_table,
}

with open(f"{RESULTS_DIR}/velocity_state_kalman_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/velocity_state_kalman_summary.json")


## Zip outputs for download from Colab

Run this cell after all figures/results have been generated.


In [ ]:
!zip -r velocity_state_kalman_outputs.zip figures results


## Control-stack status

This notebook upgrades predictive control:

```text
scalar drift estimate → velocity-state drift dynamics
```

## Interpretation

Velocity-state Kalman gives the controller a drift-rate estimate.

This creates a real predictive model, not only a lagged filtered value.

## Next steps

Possible next notebooks:

- `05_joint_parameter_filter.ipynb` — estimate Ω and B jointly with covariance.
- `06_model_predictive_control_lite.ipynb` — use predicted future drift over multiple blocks.
- `noise-mitigation/01_structured_drift_model.ipynb` — model residual drift after velocity-state control.

Guiding rule:

> Prediction requires dynamics.
